<a href="https://colab.research.google.com/github/obieshka/Python-2025-/blob/main/%D0%9F%D1%80%D0%B0%D0%BA10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn pyngrok nest-asyncio scikit-learn pandas numpy

In [2]:
import pickle
import threading
import nest_asyncio
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from fastapi import FastAPI, Query, Body
import uvicorn

nest_asyncio.apply()

housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name='MedHouseVal')

features = ['MedInc', 'AveRooms']
X = X[features]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Модель обучена и сохранена как model.pkl")

Модель обучена и сохранена как model.pkl


In [3]:
app = FastAPI(title="Housing Price Prediction API")

with open('model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

@app.get("/health")
async def health_check():
    return {"status": "ok"}

@app.get("/predict_get")
async def predict_get(MedInc: float = Query(..., description="Median Income"),
                      AveRooms: float = Query(..., description="Average Rooms")):
    input_data = np.array([[MedInc, AveRooms]])
    prediction = loaded_model.predict(input_data)[0]
    return {"prediction": round(prediction, 4)}

@app.post("/predict_post")
async def predict_post(data: dict = Body(..., example={"MedInc": 3.5, "AveRooms": 5.0})):
    try:
        MedInc = data["MedInc"]
        AveRooms = data["AveRooms"]
    except KeyError:
        return {"error": "Missing required fields: MedInc, AveRooms"}
    input_data = np.array([[MedInc, AveRooms]])
    prediction = loaded_model.predict(input_data)[0]
    return {"prediction": round(prediction, 4)}

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: FastAPIDeprecationWarning: `example` has been deprecated, please use `examples` instead
  exec(code_obj, self.user_global_ns, self.user_ns)


In [4]:
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_api, daemon=True)
thread.start()
print("API запущен на http://127.0.0.1:8000")

API запущен на http://127.0.0.1:8000


In [6]:
from pyngrok import ngrok

public_url = ngrok.connect(8000)
print(f"Публичный URL: {public_url}")
print(f"Swagger документация доступна по адресу: {public_url}/docs")

ERROR:pyngrok.process.ngrok:t=2026-08-21T11:28:48+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [7]:
import requests

# Тестируем GET
get_response = requests.get(f"{public_url}/predict_get", params={"MedInc": 4.5, "AveRooms": 6.2})
print("GET ответ:", get_response.json())

# Тестируем POST
post_response = requests.post(f"{public_url}/predict_post", json={"MedInc": 4.5, "AveRooms": 6.2})
print("POST ответ:", post_response.json())

# Health check
health_response = requests.get(f"{public_url}/health")
print("Health ответ:", health_response.json())

NameError: name 'public_url' is not defined